In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_M0_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_M0_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_M0_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_M0_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 93 trials.
Best value (Accuracy): 0.8771
Best Hyperparameters:
  outer_lr: 0.00010093304999603776
  wd: 0.0008325426470137959
  maml_inner_steps: 10
  maml_alpha_init: 0.0009888781900907544
  maml_alpha_init_eval: 0.006717556958813548
  maml_use_lslr: True
  use_lslr_at_eval: False
  use_maml_msl: hybrid
  maml_msl_num_epochs: 39
  episodes_per_epoch_train: 250
  num_experts: 24
  MOE_top_k: 7
  MOE_gate_temperature: 1.1879664247660187
  MOE_aux_coeff: 0.08672942143224953
  MOE_ctx_out_dim: 64
  MOE_ctx_hidden_dim: 32
  MOE_dropout: 0.022501513050004283
  MOE_aux_loss_plcmt: outer


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

params_to_plot_OPT = [
    "outer_lr", "wd", #"label_smooth", "ft_lr"
]

params_to_plot_MAML = [
    "maml_inner_steps", "maml_alpha_init_eval", "maml_use_lslr", "use_lslr_at_eval", "use_maml_msl", "episodes_per_epoch_train"
]

params_to_plot_MOE_CORE = [
    "num_experts", "MOE_top_k", "MOE_aux_coeff", "MOE_gate_temperature"
]

params_to_plot_MOE_AUX = [
    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_MAML)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [14]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
64,64,0.877083,None,2026-04-20 02:37:29.985498,0 days 05:54:23.908466
86,86,0.839167,None,2026-04-20 12:24:20.330820,0 days 02:26:04.122578
79,79,0.838542,None,2026-04-20 08:32:19.120076,0 days 03:51:46.177864
82,82,0.838021,None,2026-04-20 09:43:30.696367,0 days 03:20:06.484735
70,70,0.837500,None,2026-04-20 05:52:22.652707,0 days 05:56:39.666231
50,50,0.833229,None,2026-04-19 22:24:27.887600,0 days 06:44:17.953663
80,80,0.822083,None,2026-04-20 09:00:46.191933,0 days 03:28:25.547364
67,67,0.814479,None,2026-04-20 03:21:17.279809,0 days 03:01:14.876552
51,51,0.812187,None,2026-04-19 22:30:59.702746,0 days 03:55:18.017241
41,41,0.808333,None,2026-04-19 21:31:40.997107,0 days 04:23:32.971413


In [15]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'outer_lr': 0.00010093304999603776, 'wd': 0.0008325426470137959, 'maml_inner_steps': 10, 'maml_alpha_init': 0.0009888781900907544, 'maml_alpha_init_eval': 0.006717556958813548, 'maml_use_lslr': True, 'use_lslr_at_eval': False, 'use_maml_msl': 'hybrid', 'maml_msl_num_epochs': 39, 'episodes_per_epoch_train': 250, 'num_experts': 24, 'MOE_top_k': 7, 'MOE_gate_temperature': 1.1879664247660187, 'MOE_aux_coeff': 0.08672942143224953, 'MOE_ctx_out_dim': 64, 'MOE_ctx_hidden_dim': 32, 'MOE_dropout': 0.022501513050004283, 'MOE_aux_loss_plcmt': 'outer'}
{'outer_lr': 0.00017135506539036396, 'wd': 0.0007191575466293857, 'maml_inner_steps': 10, 'maml_alpha_init': 0.002234186564248213, 'maml_alpha_init_eval': 0.00874438550866414, 'maml_use_lslr': True, 'use_lslr_at_eval': True, 'use_maml_msl': 'hybrid', 'maml_msl_num_epochs': 36, 'episodes_per_epoch_train': 250, 'num_experts': 24, 'MOE_top_k': 9, 'MOE_gate_temperature': 0.6909206767846876, 'MOE_aux_coeff': 0.16382789345409485, 'MOE_ctx_out_dim': 64, '